In [13]:
"""
Inspect GPT-2 checkpoint structure to determine freezing strategy
Run this first to understand your model's layer structure
"""

import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

# ============================================================================
# EDIT THIS PATH
# ============================================================================
CHECKPOINT_PATH = r"C:\Users\prash\Documents\AI\Major Project\log\bpe16-models\model_03299.pt"
# ============================================================================

# Copy model classes from training script (required to unpickle checkpoint)
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 16384
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        pos_emb = self.transformer.wpe(pos)
        tok_emb = self.transformer.wte(idx)
        x = tok_emb + pos_emb
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

# Register in __main__ namespace for pickle to find them
import __main__
__main__.GPTConfig = GPTConfig
__main__.CausalSelfAttention = CausalSelfAttention
__main__.MLP = MLP
__main__.Block = Block
__main__.GPT = GPT

# ============================================================================

def inspect_checkpoint():
    """Load and analyze checkpoint structure"""
    
    print("=" * 80)
    print("LOADING CHECKPOINT")
    print("=" * 80)
    
    try:
        checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
        print(f"✓ Checkpoint loaded successfully from: {CHECKPOINT_PATH}\n")
    except Exception as e:
        print(f"✗ Error loading checkpoint: {e}")
        return
    
    # Display checkpoint keys
    print("Checkpoint contains keys:", list(checkpoint.keys()))
    print()
    
    # Display config
    if 'config' in checkpoint:
        config = checkpoint['config']
        print("=" * 80)
        print("MODEL CONFIGURATION")
        print("=" * 80)
        print(f"Vocabulary size: {config.vocab_size}")
        print(f"Block size (max seq len): {config.block_size}")
        print(f"Number of layers: {config.n_layer}")
        print(f"Number of heads: {config.n_head}")
        print(f"Embedding dimension: {config.n_embd}")
        print(f"Training step: {checkpoint.get('step', 'N/A')}")
        print(f"Validation loss: {checkpoint.get('val_loss', 'N/A')}")
        print()
    
    # Analyze state_dict structure
    if 'model' in checkpoint:
        state_dict = checkpoint['model']
        
        print("=" * 80)
        print("MODEL LAYER STRUCTURE")
        print("=" * 80)
        
        # Group parameters by layer type
        embeddings = []
        transformer_blocks = []
        layer_norm_final = []
        lm_head = []
        
        for name in state_dict.keys():
            if 'wte' in name or 'wpe' in name:
                embeddings.append(name)
            elif 'transformer.h.' in name:
                transformer_blocks.append(name)
            elif 'ln_f' in name:
                layer_norm_final.append(name)
            elif 'lm_head' in name:
                lm_head.append(name)
        
        print(f"\n📦 EMBEDDINGS ({len(embeddings)} params):")
        for name in sorted(embeddings):
            shape = state_dict[name].shape
            params = state_dict[name].numel()
            print(f"  {name:50s} {str(shape):20s} {params:>12,} params")
        
        print(f"\n🧱 TRANSFORMER BLOCKS ({len(transformer_blocks)} params):")
        # Group by block number
        block_params = {}
        for name in transformer_blocks:
            block_num = int(name.split('.h.')[1].split('.')[0])
            if block_num not in block_params:
                block_params[block_num] = []
            block_params[block_num].append(name)
        
        for block_num in sorted(block_params.keys()):
            print(f"\n  Block {block_num}:")
            total_params = 0
            for name in sorted(block_params[block_num]):
                shape = state_dict[name].shape
                params = state_dict[name].numel()
                total_params += params
                layer_name = name.split(f'.h.{block_num}.')[1]
                print(f"    {layer_name:45s} {str(shape):20s} {params:>12,}")
            print(f"    {'─' * 45} {'─' * 20} {'─' * 12}")
            print(f"    {'Block total':45s} {total_params:>33,}")
        
        print(f"\n🔧 FINAL LAYER NORM ({len(layer_norm_final)} params):")
        for name in sorted(layer_norm_final):
            shape = state_dict[name].shape
            params = state_dict[name].numel()
            print(f"  {name:50s} {str(shape):20s} {params:>12,} params")
        
        print(f"\n🎯 LM HEAD ({len(lm_head)} params):")
        for name in sorted(lm_head):
            shape = state_dict[name].shape
            params = state_dict[name].numel()
            print(f"  {name:50s} {str(shape):20s} {params:>12,} params")
        
        # Calculate total parameters
        total_params = sum(p.numel() for p in state_dict.values())
        print(f"\n{'=' * 80}")
        print(f"TOTAL PARAMETERS: {total_params:,}")
        print(f"{'=' * 80}\n")
        
        # Freezing recommendations
        print("=" * 80)
        print("RECOMMENDED FREEZING STRATEGIES")
        print("=" * 80)
        print("\n🔒 CONSERVATIVE (Recommended for small dataset):")
        print("   ├─ Freeze: Embeddings + Blocks 0-8 (first 9 blocks)")
        print("   └─ Train: Blocks 9-11 + Final LayerNorm + LM Head")
        print(f"   └─ Trainable params: ~{(3 * 4 * 768 * 768 + 2 * 768 + 768 * 16384) / 1e6:.1f}M")
        
        print("\n🔓 MODERATE:")
        print("   ├─ Freeze: Embeddings + Blocks 0-5 (first 6 blocks)")
        print("   └─ Train: Blocks 6-11 + Final LayerNorm + LM Head")
        print(f"   └─ Trainable params: ~{(6 * 4 * 768 * 768 + 2 * 768 + 768 * 16384) / 1e6:.1f}M")
        
        print("\n🌟 AGGRESSIVE (More data needed):")
        print("   ├─ Freeze: Only embeddings")
        print("   └─ Train: All 12 blocks + Final LayerNorm + LM Head")
        print(f"   └─ Trainable params: ~{(12 * 4 * 768 * 768 + 2 * 768 + 768 * 16384) / 1e6:.1f}M")
        
        print("\n💡 MINIMAL (Quick experiment):")
        print("   ├─ Freeze: Everything except LM Head + Final LayerNorm")
        print("   └─ Train: Only Final LayerNorm + LM Head")
        print(f"   └─ Trainable params: ~{(2 * 768 + 768 * 16384) / 1e6:.1f}M")
        
        print("\n" + "=" * 80)
        print("With your dataset (5808 train), I recommend CONSERVATIVE or MODERATE")
        print("=" * 80 + "\n")

# Run the inspection
inspect_checkpoint()

LOADING CHECKPOINT
✓ Checkpoint loaded successfully from: C:\Users\prash\Documents\AI\Major Project\log\bpe16-models\model_03299.pt

Checkpoint contains keys: ['model', 'config', 'step', 'val_loss', 'optimizer', 'rng_state', 'cuda_rng_state']

MODEL CONFIGURATION
Vocabulary size: 16384
Block size (max seq len): 1024
Number of layers: 12
Number of heads: 12
Embedding dimension: 768
Training step: 3299
Validation loss: 3.081981897354126

MODEL LAYER STRUCTURE

📦 EMBEDDINGS (2 params):
  transformer.wpe.weight                             torch.Size([1024, 768])      786,432 params
  transformer.wte.weight                             torch.Size([16384, 768])   12,582,912 params

🧱 TRANSFORMER BLOCKS (144 params):

  Block 0:
    attn.c_attn.bias                              torch.Size([2304])          2,304
    attn.c_attn.weight                            torch.Size([2304, 768])    1,769,472
    attn.c_proj.bias                              torch.Size([768])             768
    attn.c_pro